# 02 — Data Cleaning

Dtype normalization and target labeling, via `credit_risk.data.clean_and_label`.

**Scope note:** an earlier version of this notebook also ran a blanket median/`"unknown"` imputation across
*every* raw column. That's been removed — imputing ~145 raw columns when a third of them get dropped as
leakage or noise later is wasted work. Imputation now happens once, downstream, scoped to only the columns
that survive into the model matrix (`credit_risk.models.train.prepare_model_matrix`, used at training time —
see notebook 05).

In [ ]:
from credit_risk.config import settings
from credit_risk.data import clean_and_label, load_raw_data

## Load and clean

In [ ]:
df = load_raw_data(settings.raw_data_path)
cleaned = clean_and_label(df)
print("Before:", df.shape, "-> After:", cleaned.shape)

## What `clean_and_label` does

1. Strips whitespace and normalizes every text/object column to `str` — done *before* filtering, so stray
   whitespace in `loan_status` can't silently break the target filter.
2. Filters to resolved-outcome loans (`Fully Paid`, `Charged Off`).
3. Adds `target_default` (`Charged Off → 1`, `Fully Paid → 0`).

No columns are dropped and no values are imputed at this stage — `loan_status` and every raw column is still
present for downstream stages to use.

In [ ]:
cleaned.dtypes.value_counts()

In [ ]:
cleaned[["loan_status", "target_default"]].sample(10, random_state=settings.random_seed)

Confirmed: only the two resolved statuses remain, correctly mapped.